#### Imports and Load Data

In [ ]:
import os
import sys
import libpysal
import mapclassify
import numpy as np
import pandas as pd
import geopandas as gpd
from matplotlib import cm
from esda.moran import Moran
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.colors as mcolors
import matplotlib.patches as mpatches
from scipy.stats import spearmanr, mannwhitneyu, chi2_contingency
from statsmodels.stats.proportion import proportion_confint

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import config
config.apply_plot_style()

In [ ]:
df = config.load_theft_data()
df["veh_age"] = df["year"].astype(float) - df["MODL_YR"]
df.loc[~df["veh_age"].between(-1, 40), "veh_age"] = np.nan
df["veh_age"] = df["veh_age"].clip(lower=0)      # next-model-year vehicles (age -1) count as new (age 0)
df_sp = df[df["lat_final"].notna()].copy()
tracts = config.load_tracts(city_only=True)
geoids = set(tracts["GEOID"])
df_sp = df_sp[df_sp["GEOID"].isin(geoids)]
print(f"Theft reports: {len(df):,}; with make: {df['MKE_CD'].notna().sum():,}; with model year: {df['MODL_YR'].notna().sum():,}")
print(f"In city analysis tracts: {len(df_sp):,}")

# Mean age of US light vehicles in operation. Source: Bureau of Transportation Statistics, National
# Transportation Statistics Table 1-26, https://www.bts.gov/content/average-age-automobiles-and-trucks-operation-united-states
fleet_age = pd.DataFrame(list({
    2000: 8.9, 2001: 9.0, 2002: 9.0, 2003: 9.3, 2004: 9.4, 2005: 9.5, 2006: 9.6, 2007: 9.8, 2008: 9.9,
    2009: 10.2, 2010: 10.7, 2011: 11.1, 2012: 11.3, 2013: 11.4, 2014: 11.4, 2015: 11.5, 2016: 11.6,
    2017: 11.8, 2018: 11.8, 2019: 11.9, 2020: 12.1, 2021: 12.1, 2022: 12.2}.items()), columns=["year", "mean_fleet_age"])

#### Most stolen makes and models

In [ ]:
fig, (ax_make, ax_model) = plt.subplots(1, 2, figsize=config.FIGURE_SIZE_WIDE)
n_make = df["MKE_CD"].notna().sum()
top_makes = df["MKE_CD"].value_counts().head(15)
ax_make.barh(top_makes.index[::-1], top_makes.values[::-1], color="#2166ac", alpha=0.85)
for i, (m, v) in enumerate(top_makes[::-1].items()):
    ax_make.text(v, i, f" {v / n_make * 100:.1f}%", va="center")
ax_make.set_title("Top 15 Makes")
df["make_model"] = (df["MKE_CD"] + " " + df["VEH_MODL_CD"]).where(df["MKE_CD"].notna() & df["VEH_MODL_CD"].notna())
top_models = df["make_model"].value_counts().head(15)
ax_model.barh(top_models.index[::-1], top_models.values[::-1], color="#4dac26", alpha=0.85)
for i, (m, v) in enumerate(top_models[::-1].items()):
    ax_model.text(v, i, f" {v / df['make_model'].notna().sum() * 100:.1f}%", va="center")
ax_model.set_title("Top 15 Make + Model")
for a in (ax_make, ax_model):
    a.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))
    a.set_xlabel("Stolen vehicles 2000-2022")
    a.spines[["top", "right"]].set_visible(False)
fig.suptitle("Most Stolen Vehicle Makes and Models (City of LA)")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig20_top_makes_models.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Theft share by make over time, with Kia/Hyundai test

Shares are of thefts with a recorded make. 95 % Wilson intervals are shown for Kia and Hyundai.
Test: pooled share 2021-2022 vs 2015-2019 (chi-square on counts).

In [ ]:
KIA_HYUN = ["KIA", "HYUN"]
mk = df.dropna(subset=["MKE_CD"])
year_tot = mk.groupby("year").size().reindex(config.STUDY_YEARS)
FOCUS = [m for m in mk["MKE_CD"].value_counts().head(5).index if m not in KIA_HYUN] + KIA_HYUN
cnt = mk[mk["MKE_CD"].isin(FOCUS)].groupby(["year", "MKE_CD"]).size().unstack(fill_value=0).reindex(config.STUDY_YEARS, fill_value=0)
shares = cnt.div(year_tot, axis=0) * 100

COLORS = {"TOYT": "#1b9e77", "HOND": "#d95f02", "FORD": "#7570b3", "CHEV": "#e7298a", "NISS": "#66a61e",
          "KIA": "#d6604d", "HYUN": "#922b21"}
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
for m in FOCUS:
    ax.plot(config.STUDY_YEARS, shares[m], color=COLORS.get(m, "#888"), lw=1.8, marker="o", ms=3, label=m)
    if m in KIA_HYUN:
        lo, hi = proportion_confint(cnt[m], year_tot, method="wilson")
        ax.fill_between(config.STUDY_YEARS, lo * 100, hi * 100, color=COLORS[m], alpha=0.2, lw=0)
ax.set_xlabel("Year", fontsize=18)
ax.set_ylabel("Share of thefts (%)", fontsize=18)
ax.set_title("Theft Share by Vehicle Make", pad=12, fontsize=18)
ax.set_xticks(config.STUDY_YEARS)
ax.set_xticklabels(config.STUDY_YEARS, rotation=45, ha="right", fontsize=12)
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, ncol=7, fontsize=11, loc="upper center", bbox_to_anchor=(0.5, -0.25))
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig21_make_share_over_time.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

BASE, SURGE = list(range(2015, 2020)), [2021, 2022]
rows = []
for m in KIA_HYUN:
    a, n_a = cnt.loc[BASE, m].sum(), year_tot.loc[BASE].sum()
    b, n_b = cnt.loc[SURGE, m].sum(), year_tot.loc[SURGE].sum()
    chi2, p, _, _ = chi2_contingency([[a, n_a - a], [b, n_b - b]])
    rows.append({"make": m, "share_2015_2019_%": a / n_a * 100, "share_2021_2022_%": b / n_b * 100,
                 "ratio": (b / n_b) / (a / n_a), "chi2": chi2, "p_value": p,
                 **{f"share_{y}_%": shares.loc[y, m] for y in [2019, 2020, 2021, 2022]}})
# Share of Kia/Hyundai thefts that are model years 2011-2021 (the models without immobilisers)
kh = mk[mk["MKE_CD"].isin(KIA_HYUN)]
vuln = kh.assign(v=kh["MODL_YR"].between(2011, 2021)).groupby("year")["v"].mean().mul(100)
table16 = pd.DataFrame(rows)
table16.to_csv(os.path.join(config.TABLES_DIR, "table16_kia_hyundai.csv"), index=False)
print(table16.round(4).T.to_string())
print("\nKia/Hyundai thefts that are model years 2011-2021 (%):")
print(vuln.loc[2015:].round(1).to_string())

#### Stolen vehicle age vs US fleet average

In [ ]:
stolen_age = df.dropna(subset=["veh_age"]).groupby("year")["veh_age"].agg(["mean", "median"]).reset_index()
age_cmp = stolen_age.merge(fleet_age, on="year")
age_cmp["age_gap"] = age_cmp["mean"] - age_cmp["mean_fleet_age"]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
ax1.plot(age_cmp["year"], age_cmp["mean"], color="#d6604d", lw=2.2, marker="o", label="Mean age of stolen vehicles (LA)")
ax1.plot(age_cmp["year"], age_cmp["mean_fleet_age"], color="#7f8c8d", lw=2.2, ls="--", marker="s",
         label="Mean US light-vehicle fleet age (BTS)")
ax1.set_ylabel("Vehicle age (years)", fontsize=14)
ax1.set_title("Stolen Vehicle Age vs US Fleet Average", pad=12, fontsize=16)
ax1.legend(frameon=False, fontsize=11)
ax2.bar(age_cmp["year"], age_cmp["age_gap"], color=np.where(age_cmp["age_gap"] > 0, "#d6604d", "#2166ac"), width=0.7)
ax2.axhline(0, color="#555", lw=0.9)
ax2.set_ylabel("Stolen - fleet (yr)", fontsize=14)
ax2.set_xticks(config.STUDY_YEARS)
ax2.set_xticklabels(config.STUDY_YEARS, rotation=45, ha="right", fontsize=12)
for a in (ax1, ax2):
    a.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig22_age_gap.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()
print(age_cmp.round(2).to_string(index=False))

#### Theft share by model-year category

In [ ]:
AGE_CATS = ["pre-1990", "1990-1999", "2000-2009", "2010+"]
AGE_COLORS = {"pre-1990": "#67000d", "1990-1999": "#ef8a62", "2000-2009": "#d1e5f0", "2010+": "#2166ac"}
yr_cat = (df.dropna(subset=["modl_decade"]).groupby(["year", "modl_decade"], observed=False).size().unstack(fill_value=0)
          .reindex(columns=AGE_CATS, fill_value=0))
yr_cat_pct = yr_cat.div(yr_cat.sum(axis=1), axis=0) * 100
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
ax.stackplot(yr_cat_pct.index, [yr_cat_pct[c] for c in AGE_CATS], labels=AGE_CATS,
             colors=[AGE_COLORS[c] for c in AGE_CATS], alpha=0.85)
ax.set_xlim(config.STUDY_YEARS[0], config.STUDY_YEARS[-1]); ax.set_ylim(0, 100)
ax.set_xticks(config.STUDY_YEARS)
ax.set_xticklabels(config.STUDY_YEARS, rotation=45, ha="right")
ax.yaxis.set_major_formatter(mticker.PercentFormatter())
ax.set_ylabel("Share of thefts (%)")
ax.set_title("Theft Share by Vehicle Model-Year Category", pad=12)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=4)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig23_veh_age_category_share.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()
print(yr_cat_pct.loc[[2000, 2010, 2022]].round(1).to_string())

#### Hour of theft by model-year category (known times only)

In [ ]:
dt = df[~df["time_unknown"]].dropna(subset=["hour", "modl_decade"])
cont = dt.groupby(["modl_decade", "hour"], observed=False).size().unstack(fill_value=0).reindex(AGE_CATS)
chi2, p_chi, dof, _ = chi2_contingency(cont.values)
cramers_v = np.sqrt(chi2 / (cont.values.sum() * (min(cont.shape) - 1)))
print(f"χ² = {chi2:.0f}, df = {dof}, p = {p_chi:.3g}, Cramér's V = {cramers_v:.3f}, n = {cont.values.sum():,}")
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_REGULAR)
for cat in AGE_CATS:
    h = cont.loc[cat]
    ax.plot(range(24), h / h.sum() * 100, color=AGE_COLORS[cat], lw=2.2, marker="o", ms=4, label=f"{cat} (n={h.sum():,})")
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f"{h:02d}:00" for h in range(0, 24, 2)], rotation=45, ha="right")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Share within category (%)")
ax.set_title(f"Theft Hour by Model-Year Category (Cramér's V = {cramers_v:.3f})", pad=12)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False, loc="upper center", bbox_to_anchor=(0.5, -0.28), ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig24_hour_by_age_category.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Median stolen-vehicle age by tract (+ Moran's I)

Tracts around LAX (e.g. 2772.00 and the airport tract 9800.28) have a median stolen-vehicle age of 0-1 years,
many thefts at a single coordinate and a make mix led by Dodge, Chrysler and Ford, consistent with rental or
commercial fleets rather than residents' vehicles.

In [ ]:
tract_age = df_sp.dropna(subset=["veh_age"]).groupby("GEOID")["veh_age"].agg(median_stolen_age="median", n="count")
tract_age = tract_age[tract_age["n"] >= 20].reset_index()
ta = tracts.merge(tract_age, on="GEOID", how="left")
has = ta["median_stolen_age"].notna()
clf = mapclassify.NaturalBreaks(ta.loc[has, "median_stolen_age"], k=5)
ta["_class"] = np.nan
ta.loc[has, "_class"] = clf.yb.astype(float)
colors = [plt.get_cmap("viridis")(p) for p in np.linspace(0.1, 0.9, 5)]
edges = [ta.loc[has, "median_stolen_age"].min()] + list(clf.bins)
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
ta[~has].plot(ax=ax, color="#aaaaaa", edgecolor="#888", linewidth=0.2)
ta[has].plot(ax=ax, column="_class", cmap=mcolors.ListedColormap(colors), norm=mcolors.BoundaryNorm(range(6), 5),
             linewidth=0.15, edgecolor="#bbb", zorder=3)
config.add_basemap(ax)
ax.set_axis_off()
ax.set_title("Median Age of Stolen Vehicles by Census Tract (tracts with >= 20 thefts)", pad=14)
ax.legend(handles=[mpatches.Patch(facecolor=colors[i], label=f"{edges[i]:.1f}-{edges[i+1]:.1f} yrs") for i in range(5)],
          title="Median age", loc="lower left", framealpha=0.9)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig25_veh_median_age_choropleth.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

tm = ta[has].reset_index(drop=True)
w = libpysal.weights.Queen.from_dataframe(tm, use_index=False, silence_warnings=True)
w.transform = "r"
mi = Moran(tm["median_stolen_age"].values, w, permutations=999)
print(f"Moran's I (median stolen age) = {mi.I:.3f}, p_sim = {mi.p_sim:.3f}, n = {len(tm)}")

#### Dominant make by tract

Mostly reflects the composition of the local vehicle fleet, which is not observed; shown descriptively.

In [ ]:
top5 = mk["MKE_CD"].value_counts().head(5).index.tolist()
dom = (df_sp[df_sp["MKE_CD"].isin(top5)].groupby(["GEOID", "MKE_CD"]).size().reset_index(name="n")
       .sort_values("n", ascending=False).drop_duplicates("GEOID").rename(columns={"MKE_CD": "dominant_make"}))
tmk = tracts.merge(dom, on="GEOID", how="left")
MAKE_COLORS = dict(zip(top5, plt.get_cmap("Set1")(np.linspace(0, 0.8, 5))))
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
tmk[tmk["dominant_make"].isna()].plot(ax=ax, color="#aaaaaa", edgecolor="#888", linewidth=0.2)
for m in top5:
    tmk[tmk["dominant_make"] == m].plot(ax=ax, color=MAKE_COLORS[m], edgecolor="#bbb", linewidth=0.15, alpha=0.8, zorder=3)
config.add_basemap(ax)
ax.set_axis_off()
ax.set_title("Most Frequently Stolen Make by Census Tract", pad=14)
ax.legend(handles=[mpatches.Patch(facecolor=MAKE_COLORS[m], label=m) for m in top5], loc="lower left", framealpha=0.9)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig26_veh_dominant_make_map.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Make concentration ratio by tract

Ratio = share of the make among a tract's thefts / share among all mapped thefts in the same years.
Numerator and denominator use the same records (city tracts, same period). Tracts with < 30 thefts are omitted.
`tracts_ratio_gt_1.5_expected_by_chance` is the number of tracts expected above 1.5 from sampling noise alone
(binomial counts at the city share). For makes with a small city share, many exceedances are noise.

In [ ]:
MIN_TRACT_THEFTS = 30
CHANCE_SIMS = 1000


def chance_exceedances(tot, share, sims=CHANCE_SIMS, seed=42):
    """Mean number of tracts with ratio > 1.5 if every tract had the city share (binomial counts)."""
    rng = np.random.default_rng(seed)
    n = tot.values
    return float(np.mean([((rng.binomial(n, share) / n / share) > 1.5).sum() for _ in range(sims)]))


def concentration_map(makes, years, label, fname):
    sub = df_sp[df_sp["year"].isin(years) & df_sp["MKE_CD"].notna()]
    tot = sub.groupby("GEOID").size()
    hit = sub[sub["MKE_CD"].isin(makes)].groupby("GEOID").size().reindex(tot.index, fill_value=0)
    city_share = hit.sum() / tot.sum()
    ratio = (hit / tot / city_share)[tot >= MIN_TRACT_THEFTS].rename("ratio").reset_index()
    tc = tracts.merge(ratio, on="GEOID", how="left")
    vmax = max(np.nanpercentile(tc["ratio"], 97), 1.5)
    norm = mcolors.TwoSlopeNorm(vmin=0, vcenter=1, vmax=vmax)
    fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
    tc[tc["ratio"].isna()].plot(ax=ax, color="#aaaaaa", edgecolor="#888", linewidth=0.2)
    tc.dropna(subset=["ratio"]).plot(ax=ax, column="ratio", cmap="RdBu_r", norm=norm, edgecolor="#bbb", linewidth=0.15, zorder=3)
    config.add_basemap(ax)
    ax.set_axis_off()
    ax.set_title(f"{label} Theft Concentration Ratio ({years[0]}-{years[-1]}; city share {city_share*100:.1f}%)", pad=14)
    plt.colorbar(cm.ScalarMappable(cmap="RdBu_r", norm=norm), ax=ax, fraction=0.03, pad=0.02, shrink=0.75).set_label(
        "Tract share / city share (1 = city average)")
    plt.tight_layout()
    plt.savefig(os.path.join(config.FIGURES_DIR, fname), dpi=config.FIGURE_DPI, bbox_inches="tight")
    plt.close(fig)
    w = libpysal.weights.Queen.from_dataframe(tc.dropna(subset=["ratio"]).reset_index(drop=True), use_index=False, silence_warnings=True)
    w.transform = "r"
    m = Moran(tc["ratio"].dropna().values, w, permutations=999)
    return {"make": label, "years": f"{years[0]}-{years[-1]}", "city_share_%": city_share * 100,
            "n_tracts": int(tc["ratio"].notna().sum()), "tracts_ratio_gt_1.5": int((tc["ratio"] > 1.5).sum()),
            "tracts_ratio_gt_1.5_expected_by_chance": chance_exceedances(tot[tot >= MIN_TRACT_THEFTS], city_share),
            "moran_I": m.I, "moran_p": m.p_sim}


conc_rows = [concentration_map([m], config.STUDY_YEARS, m, f"fig27_veh_{m}_concentration_map.png")
             for m in ["TOYT", "HOND", "CHEV", "FORD", "NISS", "HYUN"]]
conc_rows.append(concentration_map(KIA_HYUN, SURGE, "KIA+HYUN", "fig27_veh_KIAHYUN_2021_2022_concentration_map.png"))
table17 = pd.DataFrame(conc_rows)
table17.to_csv(os.path.join(config.TABLES_DIR, "table17_make_concentration.csv"), index=False)
print(table17.round(3).to_string(index=False))

#### Vehicle age in hotspot vs other tracts (tract-level comparison)

Units are tracts (median stolen-vehicle age per tract), not individual thefts, to avoid pseudo-replication.

In [ ]:
tracts_gi = gpd.read_file(os.path.join(config.DATA_DIR, "tracts_plot_gi.gpkg"))[["GEOID", "is_hotspot", "is_hotspot_p05"]]
tracts_gi["GEOID"] = tracts_gi["GEOID"].astype(str).str.zfill(11)
tag = tract_age.merge(tracts_gi, on="GEOID")
for col in ["is_hotspot", "is_hotspot_p05"]:
    tag[col] = tag[col].astype(bool)
    h, o = tag.loc[tag[col], "median_stolen_age"], tag.loc[~tag[col], "median_stolen_age"]
    u_, p_ = mannwhitneyu(h, o, alternative="two-sided")
    print(f"[{col}] tract median stolen age: hotspots {h.median():.1f} (n={len(h)}) vs other {o.median():.1f} "
          f"(n={len(o)}); Mann-Whitney U={u_:.0f}, p={p_:.3g}")
# Figure uses the larger uncorrected set (FDR set is too small for a histogram)
hot, other = tag.loc[tag["is_hotspot_p05"], "median_stolen_age"], tag.loc[~tag["is_hotspot_p05"], "median_stolen_age"]
u, p_mw = mannwhitneyu(hot, other, alternative="two-sided")
fig, ax = plt.subplots(figsize=(10, 5))
bins = np.arange(0, 26, 0.5)
ax.hist(other, bins=bins, density=True, color="#2166ac", alpha=0.5, label=f"Other tracts (n={len(other)})")
ax.hist(hot, bins=bins, density=True, color="#c0392b", alpha=0.5, label=f"Hotspot tracts, HH p<=0.05 (n={len(hot)})")
ax.axvline(other.median(), color="#2166ac", lw=2, ls="--")
ax.axvline(hot.median(), color="#c0392b", lw=2, ls="--")
ax.set_xlabel("Tract median age of stolen vehicles (years)")
ax.set_ylabel("Density")
ax.set_title(f"Median Stolen-Vehicle Age: Hotspot vs Other Tracts (Mann-Whitney p = {p_mw:.3g})", pad=12)
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig28_veh_age_hotspot_hist.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()

#### Where are old vs new vehicles stolen? (same theft years)

Comparing old and new vehicles across the whole period confounds vehicle age with theft year (old cars were
stolen mostly in the 2000s). Here both groups come from the same theft years, 2010-2017 (stored coordinates):
model year < 2000 vs model year >= 2010.

In [ ]:
grid = config.city_grid()
g = gpd.GeoDataFrame(df_sp, geometry=gpd.points_from_xy(df_sp["lon_final"], df_sp["lat_final"]), crs=config.LATLON_CRS).to_crs(config.SPATIAL_CRS)
win = g["year"].between(2010, 2017)
old, new = g[win & (g["MODL_YR"] < 2000)], g[win & (g["MODL_YR"] >= 2010)]
z_old = config.kde_surface(old.geometry.x, old.geometry.y, grid)
z_new = config.kde_surface(new.geometry.x, new.geometry.y, grid)
d = z_new - z_old
vabs = np.nanpercentile(np.abs(d), 99)
fig, ax = plt.subplots(figsize=config.FIGURE_SIZE_SQUARE)
ax.set_xlim(grid["extent"][:2]); ax.set_ylim(grid["extent"][2:])
config.add_basemap(ax, alpha=0.6)
im = ax.imshow(np.ma.masked_where(~(np.abs(d) >= vabs * 0.05), d), origin="lower", extent=grid["extent"], cmap="RdBu_r",
               norm=mcolors.TwoSlopeNorm(vmin=-vabs, vcenter=0, vmax=vabs), alpha=0.7, zorder=3, interpolation="nearest")
ax.set_axis_off()
ax.set_title(f"New (MY>=2010, n={len(new):,}) minus Old (MY<2000, n={len(old):,}) Vehicle Thefts, 2010-2017", pad=14)
plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02, shrink=0.6).set_label("Difference in share of thefts per cell")
plt.tight_layout()
plt.savefig(os.path.join(config.FIGURES_DIR, "fig29_kde_old_vs_new.png"), dpi=config.FIGURE_DPI, bbox_inches="tight")
plt.show()
print(f"Share of density redistributed between old and new: {np.nansum(np.abs(d)) / 2 * 100:.1f}%")